# What's in a Name — Endonyms and Exonyms

What do languages call themselves, and what does English call them? This notebook
explores cross-lingual naming via `db.language_names` and each language's
`.endonym` property.

**Requirements**
```
pip install "languages-of-the-world[examples]"
```

In [ ]:
%pip install -q pandas plotly matplotlib


In [ ]:
import pandas as pd
import plotly.express as px

import low

## 1 · Load the database

In [ ]:
db = low.LanguagesOfTheWorld()
print(db)
print(f"Canonical name records: {len(db.language_names):,}")

## 2 · Endonym coverage

An endonym is a name expressed in the language itself (`LanguageName.is_endonym`).

In [ ]:
with_endonym = sum(1 for lang in db.languages if lang.endonym)
total = len(db.languages)
print(f"Languages with a known endonym: {with_endonym:,} / {total:,} ({100 * with_endonym / total:.1f}%)")

for part3 in ("deu", "hin", "kin", "jpn"):
    lang = db.languages.get(part3)
    endo = lang.endonym.name if lang.endonym else "—"
    print(f"  {lang.label} ({part3}): endonym = {endo}")

## 3 · English exonyms via `in_language`

In [ ]:
english_names = db.language_names.in_language("en")
print(f"English name records: {len(english_names):,}")

en_df = pd.DataFrame([
    {
        "part3": n.language.part3,
        "language": n.language.label,
        "english_name": n.name,
        "endonym": n.language.endonym.name if n.language.endonym else None,
    }
    for n in english_names
]).drop_duplicates(subset=["part3"])

en_df.head(10)

## 4 · Coverage bar chart

In [ ]:
coverage_rows = [
    {"category": "Has endonym", "count": with_endonym},
    {"category": "No endonym", "count": total - with_endonym},
]
cov_df = pd.DataFrame(coverage_rows)

fig = px.bar(
    cov_df,
    x="category",
    y="count",
    text="count",
    labels={"count": "Languages", "category": ""},
    title="Endonym Coverage Across All Languages",
    color="category",
    color_discrete_sequence=["#4c78a8", "#e45756"],
)
fig.update_traces(texttemplate="%{text:,}", textposition="outside")
fig.update_layout(showlegend=False, margin=dict(l=10, r=10, t=40, b=10))
fig.show()

## 5 · What is German called in French?

Use `for_language` to list all canonical names for one language.

In [ ]:
deu = db.languages.get("deu")
print(f"All canonical names for {deu.label}:")
for n in deu.names:
    endo_tag = " (endonym)" if n.is_endonym else ""
    print(f"  [{n.in_language_bcp47}] {n.name}{endo_tag}")

french_names = [n.name for n in deu.names if n.in_language_bcp47 == "fr"]
print(f"\nFrench exonym(s): {french_names}")

## 6 · European language name matrix

In [ ]:
EUROPEAN = ["deu", "fra", "ita", "spa", "eng", "nld", "pol", "rus"]
MATRIX_LANGS = ["en", "de", "fr"]

matrix_rows = []
for part3 in EUROPEAN:
    lang = db.languages.get(part3)
    row = {"language": lang.endonym.name if lang.endonym else lang.label}
    for bcp47 in MATRIX_LANGS:
        names = [n.name for n in lang.names if n.in_language_bcp47 == bcp47]
        row[bcp47] = names[0] if names else "—"
    matrix_rows.append(row)

matrix_df = pd.DataFrame(matrix_rows)
matrix_df

## 7 · Bulk endonym list

In [ ]:
endonyms = db.language_names.endonyms()
print(f"Total endonym records: {len(endonyms):,}")

endo_df = pd.DataFrame([
    {"part3": n.language.part3, "language": n.language.label, "endonym": n.name}
    for n in endonyms
]).drop_duplicates(subset=["part3"]).sort_values("language")

endo_df.head(15)

## 8 · Summary

In [ ]:
print(f"Languages with English name: {en_df['part3'].nunique():,}")
print(f"Languages with endonym: {with_endonym:,}")
both = en_df.merge(endo_df, on="part3", how="inner")
print(f"Languages with both English name and endonym: {len(both):,}")